# 01 — Exploratory Data Analysis

**Phase 2.** Everything here produces a figure or a number that belongs in the
paper. Figures are written to `/kaggle/working/artifacts/figures/` at 300 dpi.

There is one section in this notebook that is not decorative: **§7 near-duplicate
detection**. This corpus is built from templated reports, so a substantial number
of studies share near-identical text. If duplicates straddle your train/test
split, your BLEU score measures nothing. Read that section carefully.

In [ ]:
import json
import re
import collections
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["savefig.bbox"] = "tight"

ART = Path("/kaggle/working/artifacts")
FIG = ART / "figures"
FIG.mkdir(parents=True, exist_ok=True)

def save(name):
    plt.savefig(FIG / f"{name}.png")
    plt.savefig(FIG / f"{name}.pdf")
    plt.show()

## 1. Load artifacts from notebook 00

On Kaggle, chain notebooks by adding notebook 00's output as a dataset input to
this one, or just run 00 first in the same session. The loader handles both.

In [ ]:
def find_artifacts():
    local = Path("/kaggle/working/artifacts/master.csv")
    if local.exists():
        return local.parent
    for p in Path("/kaggle/input").rglob("master.csv"):
        return p.parent
    raise FileNotFoundError("Run 00_dataset_build first, or attach its output.")

SRC = find_artifacts()
print("Reading artifacts from", SRC)

df = pd.read_csv(SRC / "master.csv")
df["labels"] = df["labels"].fillna("").map(lambda s: [x for x in s.split("|") if x])
df["sections"] = df["sections_json"].map(json.loads)
multihot = pd.read_csv(SRC / "labels_multihot.csv", index_col=0)
grouped = pd.read_csv(SRC / "labels_grouped.csv", index_col=0)
ONTOLOGY = json.load(open(SRC / "ontology.json"))
tiers = json.load(open(SRC / "label_tiers.json"))

print(f"{len(df)} studies, {multihot.shape[1]} fine labels")

## 2. Disease distribution

**Figure 1 of the paper.** The long tail is the single most important fact about
this dataset and it should be the first thing a reader sees.

In [ ]:
pos = multihot.sum().sort_values(ascending=False)
colors = ["#2a6f97" if p >= 10 else "#89c2d9" if p >= 5 else "#d9d9d9" for p in pos]

fig, ax = plt.subplots(figsize=(9, 9))
ax.barh(range(len(pos)), pos.values, color=colors)
ax.set_yticks(range(len(pos)))
ax.set_yticklabels(pos.index, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Number of studies")
ax.set_title("Disease label distribution (n = %d studies)" % len(df))
for i, v in enumerate(pos.values):
    ax.text(v + 0.7, i, str(v), va="center", fontsize=8)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ["#2a6f97", "#89c2d9", "#d9d9d9"]]
ax.legend(handles, ["≥10 positives (trainable)", "5–9 (marginal)", "<5 (reported only)"],
          loc="lower right", frameon=True)
save("fig01_label_distribution")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
gp = grouped.sum().sort_values(ascending=False)
axes[0].bar(gp.index, gp.values, color="#2a6f97")
axes[0].set_title("Organ-system groups")
axes[0].tick_params(axis="x", rotation=45)
for t in axes[0].get_xticklabels():
    t.set_ha("right")

et = df["exam_type"].value_counts()
axes[1].bar(et.index, et.values, color="#e07a5f")
axes[1].set_title("Examination protocol")
axes[1].tick_params(axis="x", rotation=45)
for t in axes[1].get_xticklabels():
    t.set_ha("right")
plt.tight_layout()
save("fig02_groups_and_exam_types")

## 3. Label co-occurrence

Justifies the multi-label formulation. If the matrix were diagonal you would use
softmax; it is not, so you use BCE.

In [ ]:
top = pos[pos >= 3].index.tolist()
M = multihot[top]
co = np.asarray(M.T.values @ M.values, dtype=int).copy()
np.fill_diagonal(co, 0)
co = pd.DataFrame(co, index=top, columns=top)

plt.figure(figsize=(11, 9))
sns.heatmap(co, annot=True, fmt="d", cmap="Blues", cbar_kws={"label": "co-occurring studies"},
            linewidths=.4, annot_kws={"size": 7})
plt.title("Label co-occurrence (labels with ≥3 positives)")
save("fig03_cooccurrence")

print(f"studies with >1 label: {(multihot.sum(axis=1) > 1).sum()} "
      f"({100 * (multihot.sum(axis=1) > 1).mean():.1f}%)")

## 4. Report length

In [ ]:
df["n_words"] = df["report_clean"].str.split().str.len()
df["n_chars"] = df["report_clean"].str.len()
df["n_sentences"] = df["report_clean"].map(lambda t: len(re.findall(r"[.!?]+", t)))
df["imp_words"] = df["impression"].fillna("").str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df["n_words"], bins=30, color="#2a6f97", edgecolor="white")
axes[0].set_title("Full report length (words)")
axes[0].axvline(df["n_words"].median(), color="crimson", ls="--",
                label=f"median {df['n_words'].median():.0f}")
axes[0].legend()

axes[1].hist(df["imp_words"], bins=25, color="#e07a5f", edgecolor="white")
axes[1].set_title("Impression length (words)")
axes[1].axvline(df["imp_words"].median(), color="crimson", ls="--",
                label=f"median {df['imp_words'].median():.0f}")
axes[1].legend()

order = df.groupby("exam_type")["n_words"].median().sort_values().index
sns.boxplot(data=df, y="exam_type", x="n_words", order=order, ax=axes[2], color="#89c2d9")
axes[2].set_title("Length by protocol")
axes[2].set_ylabel("")
plt.tight_layout()
save("fig04_report_lengths")

print(df[["n_words", "n_chars", "n_sentences", "imp_words"]].describe().round(1))

## 5. Vocabulary

The token budget for the generation model comes from here. Note the vocabulary
is small and highly repetitive — that is why retrieval works so well on this
corpus, and it is worth saying so explicitly in the Discussion.

In [ ]:
STOP = set("""a an the is are was were be been being of in on at to for with and or
no not none any all both each few more most other some such only own same so than
too very can will just it its this that these those there here from by as if then
seen noted appear appears appeared shows show shown within about""".split())

tokens = [w for t in df["report_clean"] for w in re.findall(r"[A-Za-z][A-Za-z\-]+", t.lower())]
content = [w for w in tokens if w not in STOP and len(w) > 2]
vocab = collections.Counter(content)

print(f"total tokens        : {len(tokens):,}")
print(f"unique tokens       : {len(set(tokens)):,}")
print(f"type/token ratio    : {len(set(tokens)) / len(tokens):.4f}")
print(f"tokens covering 90% : "
      f"{np.searchsorted(np.cumsum([c for _, c in vocab.most_common()]) / len(content), 0.90) + 1}")

fig, ax = plt.subplots(figsize=(9, 8))
top40 = vocab.most_common(40)
ax.barh([w for w, _ in top40][::-1], [c for _, c in top40][::-1], color="#2a6f97")
ax.set_xlabel("Frequency")
ax.set_title("40 most frequent content words")
save("fig05_word_frequency")

In [ ]:
try:
    from wordcloud import WordCloud
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    wc_all = WordCloud(width=900, height=500, background_color="white",
                       colormap="viridis").generate_from_frequencies(vocab)
    axes[0].imshow(wc_all)
    axes[0].axis("off")
    axes[0].set_title("Full reports")

    imp_tokens = [w for t in df["impression"].fillna("")
                  for w in re.findall(r"[A-Za-z][A-Za-z\-]+", t.lower())
                  if w not in STOP and len(w) > 2]
    wc_imp = WordCloud(width=900, height=500, background_color="white",
                       colormap="plasma").generate_from_frequencies(collections.Counter(imp_tokens))
    axes[1].imshow(wc_imp)
    axes[1].axis("off")
    axes[1].set_title("Impressions only")
    plt.tight_layout()
    save("fig06_wordclouds")
except ImportError:
    print("wordcloud not installed — `pip install wordcloud`. Skipping (optional figure).")

## 6. Section structure

Evidence for the section-aware encoding contribution: sections are present in a
protocol-dependent pattern, so a flat bag-of-text encoder throws away structure
that is actually informative about *where* to look in the image.

In [ ]:
sec_counts = collections.Counter(k for s in df["sections"] for k in s)
common_secs = [s for s, c in sec_counts.most_common(20)]

presence = pd.DataFrame(
    [[int(s in secs) for s in common_secs] for secs in df["sections"]],
    columns=common_secs, index=df.index,
)
presence["exam_type"] = df["exam_type"].values
heat = presence.groupby("exam_type")[common_secs].mean()

plt.figure(figsize=(13, 4.5))
sns.heatmap(heat, cmap="YlGnBu", annot=False, cbar_kws={"label": "fraction of studies"},
            linewidths=.3)
plt.title("Section presence by examination protocol")
plt.ylabel("")
save("fig07_section_presence")

print("Mean sections per report:", round(df["n_sections"].mean(), 2))
print("\nSection frequency:")
for s, c in sec_counts.most_common(20):
    print(f"  {s:22s} {c:4d}  ({100*c/len(df):5.1f}%)")

## 7. Near-duplicate detection — read this before splitting the data

Reports are generated from templates, so two studies with different diagnoses can
share 85% of their text. Two consequences:

1. **Generation metrics are inflated.** If a near-twin of a test report is in the
   training set, copying it verbatim scores high BLEU while demonstrating nothing.
2. **Retrieval looks artificially good.** Recall@1 climbs because the index
   contains a near-copy.

The fix is not to delete duplicates — it is to keep near-duplicate clusters
**together inside a single fold**, which notebook 02 does via group-aware splitting.
This section produces the cluster assignment that notebook 02 consumes.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X = vec.fit_transform(df["report_clean"])
S = cosine_similarity(X)
np.fill_diagonal(S, 0)

tri = S[np.triu_indices_from(S, k=1)]
plt.figure(figsize=(8, 4))
plt.hist(tri, bins=60, color="#2a6f97", edgecolor="white")
for thr, c in [(0.80, "crimson"), (0.90, "darkred")]:
    plt.axvline(thr, color=c, ls="--", label=f"{thr:.2f}  ({(tri > thr).sum()} pairs)")
plt.xlabel("TF-IDF cosine similarity between report pairs")
plt.ylabel("Pair count")
plt.title("Pairwise report similarity")
plt.legend()
save("fig08_report_similarity")

print(f"pairs > 0.95 (effectively identical): {(tri > 0.95).sum()}")
print(f"pairs > 0.90                        : {(tri > 0.90).sum()}")
print(f"pairs > 0.80                        : {(tri > 0.80).sum()}")
print(f"mean similarity                     : {tri.mean():.3f}")

### Choosing the grouping threshold

Grouping is transitive-closure over the similarity graph, so a low threshold
chains unrelated studies into one giant component. Sweep first, then pick the
highest threshold that still catches the template families — and report the sweep
in the paper so the choice is not arbitrary.

In [ ]:
def sweep(threshold):
    par = list(range(len(df)))

    def fnd(a):
        while par[a] != a:
            par[a] = par[par[a]]
            a = par[a]
        return a

    ii, jj = np.where(np.triu(S, 1) > threshold)
    for a, b in zip(ii, jj):
        ra, rb = fnd(int(a)), fnd(int(b))
        if ra != rb:
            par[max(ra, rb)] = min(ra, rb)
    g = pd.Series([fnd(i) for i in range(len(df))])
    sz = g.value_counts()
    return {"threshold": threshold, "n_groups": g.nunique(),
            "largest_group": int(sz.max()), "groups_gt1": int((sz > 1).sum())}

sweep_df = pd.DataFrame([sweep(t) for t in [0.80, 0.85, 0.90, 0.93, 0.95, 0.97]])
sweep_df.to_csv(ART / "dup_threshold_sweep.csv", index=False)
print(sweep_df.to_string(index=False))

In [ ]:
DUP_THRESHOLD = 0.93

parent = list(range(len(df)))

def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)

ii, jj = np.where(np.triu(S, 1) > DUP_THRESHOLD)
for a, b in zip(ii, jj):
    union(int(a), int(b))

df["dup_group"] = [find(i) for i in range(len(df))]
sizes = df["dup_group"].value_counts()

print(f"threshold          : {DUP_THRESHOLD}")
print(f"unique groups      : {df['dup_group'].nunique()}  (from {len(df)} studies)")
print(f"groups with >1     : {(sizes > 1).sum()}")
print(f"largest group size : {sizes.max()}")
print()
print("Largest near-duplicate clusters:")
for g in sizes.head(5).index:
    members = df.loc[df["dup_group"] == g]
    print(f"\n  group {g} — {len(members)} studies: {members['study_id'].tolist()}")
    print(f"    labels: {sorted({l for ls in members['labels'] for l in ls})}")
    print(f"    sample impression: {members['impression'].iloc[0][:100]}")

df[["study_id", "dup_group"]].to_csv(ART / "dup_groups.csv", index=False)
print("\nSaved dup_groups.csv — notebook 02 uses this for leakage-free splits.")

## 8. Image statistics

These images are photographs/exports of ultrasound console screens, each
containing **several stacked frames**. Establishing that here motivates the panel
decomposition step in notebook 02.

In [ ]:
from PIL import Image

img_df = df[df["has_image"]].copy()
rows = []
for sid, p in zip(img_df["study_id"], img_df["image_path"]):
    try:
        with Image.open(p) as im:
            w, h = im.size
            rows.append({"study_id": sid, "width": w, "height": h,
                         "aspect": w / h, "megapixels": w * h / 1e6, "mode": im.mode})
    except Exception as e:
        rows.append({"study_id": sid, "width": np.nan, "height": np.nan,
                     "aspect": np.nan, "megapixels": np.nan, "mode": f"ERROR: {e}"})

imeta = pd.DataFrame(rows)
imeta.to_csv(ART / "image_metadata.csv", index=False)
print(imeta[["width", "height", "aspect", "megapixels"]].describe().round(2))
print("\nmodes:", imeta["mode"].value_counts().to_dict())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(imeta["width"], imeta["height"], alpha=.5, s=18, color="#2a6f97")
axes[0].set_xlabel("width (px)")
axes[0].set_ylabel("height (px)")
axes[0].set_title("Image dimensions")
axes[1].hist(imeta["aspect"].dropna(), bins=30, color="#e07a5f", edgecolor="white")
axes[1].set_title("Aspect ratio (w/h)")
axes[1].axvline(1.0, color="k", ls=":")
axes[2].hist(imeta["megapixels"].dropna(), bins=30, color="#81b29a", edgecolor="white")
axes[2].set_title("Resolution (MP)")
plt.tight_layout()
save("fig09_image_stats")

In [ ]:
sample = img_df.sample(min(8, len(img_df)), random_state=0)
fig, axes = plt.subplots(2, 4, figsize=(16, 9))
for ax, (_, r) in zip(axes.ravel(), sample.iterrows()):
    try:
        ax.imshow(Image.open(r["image_path"]))
    except Exception:
        pass
    ax.set_title(f"{r['study_id']}\n{r['primary_label'][:28]}", fontsize=9)
    ax.axis("off")
plt.suptitle("Sample studies — note the multi-frame composite layout", y=1.0)
plt.tight_layout()
save("fig10_sample_images")

## 9. Data quality summary

In [ ]:
quality = {
    "studies": len(df),
    "missing_image": int((~df["has_image"]).sum()),
    "missing_impression": int((df["impression"].fillna("").str.len() < 3).sum()),
    "unlabelled": int((df["primary_label"] == "Unlabelled").sum()),
    "exact_duplicate_reports": int(df["report_clean"].duplicated().sum()),
    "near_duplicate_pairs_085": int((tri > 0.85).sum()),
    "near_duplicate_groups": int((sizes > 1).sum()),
    "labels_with_ge10_pos": len(tiers["trainable"]),
    "labels_with_lt5_pos": len(tiers["rare"]),
    "vocab_size": len(set(tokens)),
    "median_report_words": int(df["n_words"].median()),
    "median_impression_words": int(df["imp_words"].median()),
}
json.dump(quality, open(ART / "eda_summary.json", "w"), indent=2)
print(json.dumps(quality, indent=2))

df.drop(columns=["sections", "labels"]).to_csv(ART / "master_eda.csv", index=False)
print("\nFigures written to", FIG)

---
### What to carry into the paper

* **Fig. 1** (label distribution) — justifies why you evaluate on a label subset.
* **Fig. 3** (co-occurrence) — justifies the multi-label formulation.
* **Fig. 7** (section presence) — motivates section-aware encoding.
* **Fig. 8** (report similarity) — this is your methodological honesty figure.
  Very few small-dataset radiology papers report it, and it will make a reviewer
  trust the rest of your numbers.

Next: **02_preprocessing_and_splits.ipynb**.